In [1]:
import xarray as xr
import pandas as pd
import os

In [2]:
# Define a function to convert DMS to decimal degrees
def dms_to_decimal(degrees, minutes, seconds=0):
    """
    Convert degrees, minutes, and seconds to decimal degrees.
    
    Parameters:
    - degrees: The degrees part of the coordinate
    - minutes: The minutes part of the coordinate
    - seconds: The seconds part of the coordinate (optional)
    
    Returns:
    - Decimal degrees as a float
    """
    return degrees + (minutes / 60) + (seconds / 3600)

# Longitude: 84°25′ (84 degrees, 25 minutes, 0 seconds)
target_lon = dms_to_decimal(84, 25)

# Latitude: 44°22′ (44 degrees, 22 minutes, 0 seconds)
target_lat = dms_to_decimal(44, 22)


target_lat, target_lon = 41.56667,79.61667 # Location of xhl source: https://www.docin.com/p-2020786258.html

In [6]:
variable_list = ['pr', 'tas', 'tasmax', 'tasmin']
combined_df_dict = {'pr': None, 'tas': None, 'tasmax': None, 'tasmin': None}
for variable in variable_list:
    # Directory where your NetCDF files are stored
    directory_path = "F:\\geodata\\CMIP6\\"+variable
    
    # List all the NetCDF files in the directory
    # Assuming the files follow the naming convention 'tas_day_MIROC6...'
    
    file_names = os.listdir(directory_path)
    # Initialize an empty list to store DataFrames for each file
    df_list = []
    
    # Loop through the list of NetCDF files
    for file_name in file_names:
        # Construct full file path
        file_path = os.path.join(directory_path, file_name)
        
        # Load the NetCDF dataset
        ds = xr.open_dataset(file_path)
        
        # Print dataset information (optional, useful to inspect variables and coordinates)
        # print(ds)
        
        # Find the nearest longitude and latitude in the dataset
        # Assuming the coordinates are named 'lon' and 'lat' in the dataset
        nearest_point = ds.sel(lon=target_lon, lat=target_lat, method='nearest')
        
        # Extract the time series for the 'tas' variable (temperature in Kelvin)
        variable_series = nearest_point[variable]
        
        # Convert temperature to Celsius (tas is usually in Kelvin)
        if variable != 'pr':
            variable_series = variable_series - 272.15

        
        # Convert the xarray DataArray to a Pandas DataFrame
        df_variable_series = variable_series.to_dataframe().reset_index()
        
        # Append the DataFrame to the list
        df_list.append(df_variable_series)
    
    # Concatenate all the DataFrames from different files into one
    df_combined = pd.concat(df_list, ignore_index=True)
    df_combined.set_index('time', inplace=True)
    # Optionally, inspect the combined DataFrame
    combined_df_dict[variable] = df_combined[variable]

In [20]:
future_df= pd.DataFrame(combined_df_dict)
future_df.columns = ['pre','tm','tmax','tmin']
# future_df['time'] = pd.to_datetime(future_df.index, format='Y-MM-DD')
future_df.index = future_df.index.normalize()
future_df.head()

,pre,tm,tmax,tmin
time,,,,
2015-01-01,1.352112e-01,-12.094421,-9.422363,-15.239777
2015-01-02,2.255587e-01,-11.622070,-9.600861,-13.228241
2015-01-03,1.359639e-02,-9.292969,-4.817932,-13.446472
2015-01-04,1.298427e-15,-4.139374,-1.100891,-7.110321
2015-01-05,5.189253e-15,-6.173279,-3.788391,-8.430725


In [21]:
future_df.to_csv(os.path.join("F:\geodata\CMIP6",'day_MIROC6_ssp245_r1i1p1f1_gn.csv'))

In [22]:
history_df = pd.read_csv(r"F:\geodata\river_runoff_obs\3_xhl_imputMF.csv")
# Convert 'time' column to datetime
history_df['time'] = pd.to_datetime(history_df['time'])
# Set 'time' as the index
history_df.set_index('time', inplace=True)
history_df.tail()

,pre,tm,tmax,tmin,dis,mon
time,,,,,,
2022-12-27,0.099489,-12.392940,-4.050647,-18.807302,NaN,2658.999648
2022-12-28,0.074695,-12.987122,-5.739622,-19.052650,NaN,2658.999648
2022-12-29,0.052963,-12.244612,-5.046828,-18.020178,NaN,2658.999648
2022-12-30,0.058756,-11.706182,-5.572570,-16.590723,NaN,2658.999648
2022-12-31,0.053235,-12.114544,-4.904123,-17.854440,NaN,2658.999648


In [23]:
future_2023_df = future_df[future_df.index>='2023-01-01']
future_2023_df.head()

,pre,tm,tmax,tmin
time,,,,
2023-01-01,2.944362,-1.741699,0.935394,-4.229553
2023-01-02,2.104665,-5.177124,-3.972931,-6.720490
2023-01-03,0.393959,-7.145264,-5.058441,-8.649536
2023-01-04,0.226951,-7.280121,-5.139587,-9.037964
2023-01-05,0.175483,-8.726562,-5.758331,-12.723480


In [25]:
df_concat = pd.concat([history_df, future_2023_df])
df_concat.to_csv(r"F:\geodata\river_runoff_obs\3_xhl_imputMF_future.csv")